In [1]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

In [2]:
DATA_PATH = "../data/processed/cleaned_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (2830743, 79)


In [3]:
PREPROCESSING_DIR = "../models/preprocessing"

scaler = joblib.load(
    os.path.join(PREPROCESSING_DIR, "scaler.joblib")
)

label_encoder = joblib.load(
    os.path.join(PREPROCESSING_DIR, "label_encoder.joblib")
)

print("Scaler loaded:", type(scaler).__name__)
print("Label encoder loaded:", type(label_encoder).__name__)

Scaler loaded: StandardScaler
Label encoder loaded: LabelEncoder


In [4]:
FEATURES = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Length of Fwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Variance",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Std"
]

print("Number of features:", len(FEATURES))

Number of features: 45


In [5]:
X = df[FEATURES]
y = label_encoder.transform(df["Label"])

print("Feature matrix:", X.shape)
print("Target:", y.shape)

Feature matrix: (2830743, 45)
Target: (2830743,)


In [6]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (1981520, 45)
Validation: (424611, 45)
Test: (424612, 45)


In [7]:
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled training:", X_train_scaled.shape)
print("Scaled validation:", X_val_scaled.shape)
print("Scaled test:", X_test_scaled.shape)

Scaled training: (1981520, 45)
Scaled validation: (424611, 45)
Scaled test: (424612, 45)


In [8]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1
)

In [9]:
logistic_model = LogisticRegression(
    max_iter=1000,
    solver="lbfgs",
    n_jobs=-1
)

print("Starting Logistic Regression training...")

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression training completed!")

Starting Logistic Regression training...


f:\NIDS-ML-NEW\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Logistic Regression training completed!


In [10]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating Logistic Regression...")

y_val_pred = logistic_model.predict(X_val_scaled)

accuracy = accuracy_score(y_val, y_val_pred)

print("Validation Accuracy:", accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating Logistic Regression...
Validation Accuracy: 0.9744754610690727

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.98      0.98    340964
                       Bot       0.75      0.01      0.02       295
                      DDoS       1.00      0.95      0.97     19204
             DoS GoldenEye       0.95      0.82      0.88      1544
                  DoS Hulk       0.94      1.00      0.97     34661
          DoS Slowhttptest       0.82      0.80      0.81       825
             DoS slowloris       0.88      0.55      0.68       870
               FTP-Patator       0.96      1.00      0.98      1190
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       0.33      0.17      0.22         6
                  PortScan       0.82      1.00      0.90     23839
               SSH-Patator       0.97      0.47      0.64       884
  Web Attack - Br

In [11]:
MODEL_DIR = "../models"

os.makedirs(MODEL_DIR, exist_ok=True)

logistic_path = os.path.join(
    MODEL_DIR,
    "logistic_regression.joblib"
)

joblib.dump(
    logistic_model,
    logistic_path
)

print("Logistic Regression model saved successfully!")
print("Path:", logistic_path)

Logistic Regression model saved successfully!
Path: ../models\logistic_regression.joblib


In [12]:
print(
    "Model exists:",
    os.path.exists(logistic_path)
)

Model exists: True


In [13]:
from sklearn.ensemble import RandomForestClassifier

print("Random Forest imported successfully!")

Random Forest imported successfully!


In [14]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

print("Random Forest model created successfully!")

Random Forest model created successfully!


In [15]:
print("Starting Random Forest training...")

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed!")

Starting Random Forest training...
Random Forest training completed!


In [16]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating Random Forest...")

y_val_pred_rf = random_forest_model.predict(X_val)

accuracy_rf = accuracy_score(y_val, y_val_pred_rf)

print("Validation Accuracy:", accuracy_rf)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_rf,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating Random Forest...
Validation Accuracy: 0.9986481744467289

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    340964
                       Bot       0.88      0.78      0.83       295
                      DDoS       1.00      1.00      1.00     19204
             DoS GoldenEye       1.00      0.99      1.00      1544
                  DoS Hulk       1.00      1.00      1.00     34661
          DoS Slowhttptest       1.00      0.99      0.99       825
             DoS slowloris       0.99      1.00      0.99       870
               FTP-Patator       1.00      1.00      1.00      1190
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.67      0.80         6
                  PortScan       0.99      1.00      1.00     23839
               SSH-Patator       1.00      1.00      1.00       884
  Web Attack - Brute Fo

In [17]:
MODEL_DIR = "../models/supervised"

os.makedirs(MODEL_DIR, exist_ok=True)

random_forest_path = os.path.join(
    MODEL_DIR,
    "random_forest.joblib"
)

joblib.dump(
    random_forest_model,
    random_forest_path
)

print("Random Forest model saved successfully!")
print("Path:", random_forest_path)

Random Forest model saved successfully!
Path: ../models/supervised\random_forest.joblib


In [18]:
print(
    "Model exists:",
    os.path.exists(random_forest_path)
)

Model exists: True


In [1]:
from xgboost import XGBClassifier

print("XGBoost imported successfully!")

XGBoost imported successfully!


In [2]:
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=15,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

print("XGBoost model created successfully!")

XGBoost model created successfully!


In [4]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split

# Load cleaned dataset
DATA_PATH = "../data/processed/cleaned_data.csv"
df = pd.read_csv(DATA_PATH)

# Same 45 features selected in Phase 4
FEATURES = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Length of Fwd Packets",
    "Fwd Packet Length Max",
    "Fwd Packet Length Min",
    "Fwd Packet Length Mean",
    "Bwd Packet Length Max",
    "Bwd Packet Length Min",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Mean",
    "Fwd IAT Std",
    "Fwd IAT Min",
    "Bwd IAT Total",
    "Bwd IAT Mean",
    "Bwd IAT Std",
    "Bwd IAT Max",
    "Bwd IAT Min",
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "Bwd Packets/s",
    "Min Packet Length",
    "Max Packet Length",
    "Packet Length Mean",
    "Packet Length Variance",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Down/Up Ratio",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward",
    "min_seg_size_forward",
    "Active Mean",
    "Active Std",
    "Active Max",
    "Active Min",
    "Idle Std"
]

# Feature matrix
X = df[FEATURES]

# Load saved label encoder
label_encoder = joblib.load(
    "../models/preprocessing/label_encoder.joblib"
)

# Encode target
y = label_encoder.transform(df["Label"])

# Recreate the same train/temporary split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Recreate validation/test split
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Training: (1981520, 45) (1981520,)
Validation: (424611, 45) (424611,)
Test: (424612, 45) (424612,)


In [5]:
print("Starting XGBoost training...")

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost training completed!")

Starting XGBoost training...
XGBoost training completed!


In [6]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating XGBoost...")

y_val_pred_xgb = xgb_model.predict(X_val)

accuracy_xgb = accuracy_score(
    y_val,
    y_val_pred_xgb
)

print("Validation Accuracy:", accuracy_xgb)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_xgb,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating XGBoost...
Validation Accuracy: 0.9989778879963072

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       1.00      1.00      1.00    340964
                       Bot       0.93      0.70      0.80       295
                      DDoS       1.00      1.00      1.00     19204
             DoS GoldenEye       0.99      1.00      1.00      1544
                  DoS Hulk       1.00      1.00      1.00     34661
          DoS Slowhttptest       0.99      0.99      0.99       825
             DoS slowloris       0.99      1.00      0.99       870
               FTP-Patator       1.00      1.00      1.00      1190
                Heartbleed       1.00      1.00      1.00         2
              Infiltration       1.00      0.67      0.80         6
                  PortScan       0.99      1.00      1.00     23839
               SSH-Patator       1.00      1.00      1.00       884
  Web Attack - Brute Force   

In [8]:
import os
import joblib

MODEL_DIR = "../models/supervised"

os.makedirs(MODEL_DIR, exist_ok=True)

xgb_path = os.path.join(
    MODEL_DIR,
    "xgboost.joblib"
)

joblib.dump(
    xgb_model,
    xgb_path
)

print("XGBoost model saved successfully!")
print("Path:", xgb_path)
print("Model exists:", os.path.exists(xgb_path))

XGBoost model saved successfully!
Path: ../models/supervised\xgboost.joblib
Model exists: True


In [9]:
from lightgbm import LGBMClassifier

print("LightGBM imported successfully!")

LightGBM imported successfully!


In [10]:
lgbm_model = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=-1,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multiclass",
    num_class=15,
    random_state=42,
    n_jobs=-1
)

print("LightGBM model created successfully!")

LightGBM model created successfully!


In [11]:
print("Starting LightGBM training...")

lgbm_model.fit(
    X_train,
    y_train
)

print("LightGBM training completed!")

Starting LightGBM training...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.218141 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8719
[LightGBM] [Info] Number of data points in the train set: 1981520, number of used features: 45
[LightGBM] [Info] Start training from score -0.219396
[LightGBM] [Info] Start training from score -7.272439
[LightGBM] [Info] Start training from score -3.096052
[LightGBM] [Info] Start training from score -5.616844
[LightGBM] [Info] Start training from score -2.505561
[LightGBM] [Info] Start training from score -6.243806
[LightGBM] [Info] Start training from score -6.191176
[LightGBM] [Info] Start training from score -5.876561
[LightGBM] [Info] Start training from score -12.419933
[LightGBM] [Info] Start training from score -11.280499
[LightGBM] [Info] Start training from score -2.879831
[Li

In [12]:
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating LightGBM...")

y_val_pred_lgbm = lgbm_model.predict(X_val)

accuracy_lgbm = accuracy_score(
    y_val,
    y_val_pred_lgbm
)

print("Validation Accuracy:", accuracy_lgbm)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_lgbm,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

Evaluating LightGBM...
Validation Accuracy: 0.8825466132530716

Classification Report:
                            precision    recall  f1-score   support

                    BENIGN       0.91      0.97      0.94    340964
                       Bot       0.00      0.00      0.00       295
                      DDoS       0.81      0.43      0.56     19204
             DoS GoldenEye       0.12      0.04      0.06      1544
                  DoS Hulk       0.86      0.77      0.81     34661
          DoS Slowhttptest       0.00      0.00      0.00       825
             DoS slowloris       0.00      0.00      0.00       870
               FTP-Patator       0.00      0.00      0.00      1190
                Heartbleed       0.00      0.00      0.00         2
              Infiltration       0.00      0.00      0.00         6
                  PortScan       1.00      0.39      0.56     23839
               SSH-Patator       0.00      0.00      0.00       884
  Web Attack - Brute Force  

In [13]:
import os
import joblib

MODEL_DIR = "../models/supervised"

os.makedirs(MODEL_DIR, exist_ok=True)

lgbm_path = os.path.join(
    MODEL_DIR,
    "lightgbm.joblib"
)

joblib.dump(
    lgbm_model,
    lgbm_path
)

print("LightGBM model saved successfully!")
print("Path:", lgbm_path)
print("Model exists:", os.path.exists(lgbm_path))

LightGBM model saved successfully!
Path: ../models/supervised\lightgbm.joblib
Model exists: True


In [14]:
from sklearn.ensemble import IsolationForest

isolation_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

print("Isolation Forest model created successfully!")

Isolation Forest model created successfully!


In [15]:
# Convert 15-class labels into Normal vs Attack
y_train_binary = (y_train != label_encoder.transform(["BENIGN"])[0]).astype(int)
y_val_binary = (y_val != label_encoder.transform(["BENIGN"])[0]).astype(int)
y_test_binary = (y_test != label_encoder.transform(["BENIGN"])[0]).astype(int)

print("Binary labels created successfully!")

print("\nTraining set:")
print("Normal:", (y_train_binary == 0).sum())
print("Attack:", (y_train_binary == 1).sum())

print("\nValidation set:")
print("Normal:", (y_val_binary == 0).sum())
print("Attack:", (y_val_binary == 1).sum())

print("\nTest set:")
print("Normal:", (y_test_binary == 0).sum())
print("Attack:", (y_test_binary == 1).sum())

Binary labels created successfully!

Training set:
Normal: 1591168
Attack: 390352

Validation set:
Normal: 340964
Attack: 83647

Test set:
Normal: 340965
Attack: 83647


In [17]:
# Load the saved scaler
import joblib

scaler = joblib.load("../models/preprocessing/scaler.joblib")

# Scale the existing train, validation and test data
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Scaled datasets recreated successfully!")

print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

Scaled datasets recreated successfully!
X_train_scaled: (1981520, 45)
X_val_scaled: (424611, 45)
X_test_scaled: (424612, 45)


In [18]:
print("Training Isolation Forest...")

isolation_forest.fit(X_train_scaled)

print("Isolation Forest training completed!")

Training Isolation Forest...
Isolation Forest training completed!


In [19]:
# Generate Isolation Forest predictions
train_predictions = isolation_forest.predict(X_train_scaled)
val_predictions = isolation_forest.predict(X_val_scaled)
test_predictions = isolation_forest.predict(X_test_scaled)

# Convert: 1 = Normal, -1 = Anomaly
train_predictions = (train_predictions == -1).astype(int)
val_predictions = (val_predictions == -1).astype(int)
test_predictions = (test_predictions == -1).astype(int)

print("Predictions generated successfully!")

print("\nTraining predictions:")
print("Normal:", (train_predictions == 0).sum())
print("Anomaly:", (train_predictions == 1).sum())

print("\nValidation predictions:")
print("Normal:", (val_predictions == 0).sum())
print("Anomaly:", (val_predictions == 1).sum())

print("\nTest predictions:")
print("Normal:", (test_predictions == 0).sum())
print("Anomaly:", (test_predictions == 1).sum())

Predictions generated successfully!

Training predictions:
Normal: 1849945
Anomaly: 131575

Validation predictions:
Normal: 396425
Anomaly: 28186

Test predictions:
Normal: 396569
Anomaly: 28043


In [20]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Validation metrics
accuracy = accuracy_score(y_val_binary, val_predictions)
precision = precision_score(y_val_binary, val_predictions)
recall = recall_score(y_val_binary, val_predictions)
f1 = f1_score(y_val_binary, val_predictions)

print("Isolation Forest Validation Results")
print("-" * 40)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nClassification Report:")
print(classification_report(
    y_val_binary,
    val_predictions,
    target_names=["Normal", "Attack"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val_binary, val_predictions))

Isolation Forest Validation Results
----------------------------------------
Accuracy : 0.7889
Precision: 0.3940
Recall   : 0.1327
F1 Score : 0.1986

Classification Report:
              precision    recall  f1-score   support

      Normal       0.82      0.95      0.88    340964
      Attack       0.39      0.13      0.20     83647

    accuracy                           0.79    424611
   macro avg       0.61      0.54      0.54    424611
weighted avg       0.73      0.79      0.74    424611

Confusion Matrix:
[[323882  17082]
 [ 72543  11104]]


In [21]:
import os
import joblib

MODEL_DIR = "../models/anomaly"

os.makedirs(MODEL_DIR, exist_ok=True)

isolation_path = os.path.join(
    MODEL_DIR,
    "isolation_forest.joblib"
)

joblib.dump(
    isolation_forest,
    isolation_path
)

print("Isolation Forest saved successfully!")
print("Path:", isolation_path)
print("Model exists:", os.path.exists(isolation_path))

Isolation Forest saved successfully!
Path: ../models/anomaly\isolation_forest.joblib
Model exists: True


In [22]:
model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "Accuracy": [
        0.974475,
        0.998648,
        0.998978,
        0.882547
    ],
    "Macro F1": [
        0.60,
        0.85,
        0.88,
        0.20
    ]
})

model_comparison

,Model,Accuracy,Macro F1
0,Logistic Regression,0.974475,0.60
1,Random Forest,0.998648,0.85
2,XGBoost,0.998978,0.88
3,LightGBM,0.882547,0.20


In [23]:
anomaly_comparison = pd.DataFrame({
    "Model": ["Isolation Forest"],
    "Task": ["Normal vs Attack"],
    "Accuracy": [0.7889],
    "Precision": [0.3940],
    "Recall": [0.1327],
    "F1 Score": [0.1986]
})

anomaly_comparison

,Model,Task,Accuracy,Precision,Recall,F1 Score
0,Isolation Forest,Normal vs Attack,0.7889,0.394,0.1327,0.1986


In [24]:
print("=" * 70)
print("PHASE 5 — MODEL TRAINING SUMMARY")
print("=" * 70)

print("\nSUPERVISED CLASSIFICATION MODELS")
print("-" * 70)

print("1. Logistic Regression")
print("   Accuracy : 97.45%")
print("   Macro F1 : 0.60")

print("\n2. Random Forest")
print("   Accuracy : 99.86%")
print("   Macro F1 : 0.85")

print("\n3. XGBoost")
print("   Accuracy : 99.90%")
print("   Macro F1 : 0.88")

print("\n4. LightGBM")
print("   Accuracy : 88.25%")
print("   Macro F1 : 0.20")

print("\nANOMALY DETECTION")
print("-" * 70)

print("5. Isolation Forest")
print("   Task      : Normal vs Attack")
print("   Accuracy  : 78.89%")
print("   Precision : 39.40%")
print("   Recall    : 13.27%")
print("   F1 Score  : 19.86%")

print("\n" + "=" * 70)
print("PHASE 5 MODEL TRAINING COMPLETED")
print("=" * 70)

PHASE 5 — MODEL TRAINING SUMMARY

SUPERVISED CLASSIFICATION MODELS
----------------------------------------------------------------------
1. Logistic Regression
   Accuracy : 97.45%
   Macro F1 : 0.60

2. Random Forest
   Accuracy : 99.86%
   Macro F1 : 0.85

3. XGBoost
   Accuracy : 99.90%
   Macro F1 : 0.88

4. LightGBM
   Accuracy : 88.25%
   Macro F1 : 0.20

ANOMALY DETECTION
----------------------------------------------------------------------
5. Isolation Forest
   Task      : Normal vs Attack
   Accuracy  : 78.89%
   Precision : 39.40%
   Recall    : 13.27%
   F1 Score  : 19.86%

PHASE 5 MODEL TRAINING COMPLETED


In [25]:
import os

model_paths = {
    "Logistic Regression": "../models/supervised/logistic_regression.joblib",
    "Random Forest": "../models/supervised/random_forest.joblib",
    "XGBoost": "../models/supervised/xgboost.joblib",
    "LightGBM": "../models/supervised/lightgbm.joblib",
    "Isolation Forest": "../models/anomaly/isolation_forest.joblib"
}

print("MODEL FILE VERIFICATION")
print("=" * 60)

all_models_exist = True

for model_name, path in model_paths.items():
    exists = os.path.exists(path)

    print(f"{model_name:20} : {'FOUND' if exists else 'MISSING'}")
    print(f"Path                 : {path}")

    if not exists:
        all_models_exist = False

print("=" * 60)

if all_models_exist:
    print("All 5 trained models are saved successfully!")
else:
    print("Some model files are missing. Check the paths above.")

MODEL FILE VERIFICATION
Logistic Regression  : MISSING
Path                 : ../models/supervised/logistic_regression.joblib
Random Forest        : FOUND
Path                 : ../models/supervised/random_forest.joblib
XGBoost              : FOUND
Path                 : ../models/supervised/xgboost.joblib
LightGBM             : FOUND
Path                 : ../models/supervised/lightgbm.joblib
Isolation Forest     : FOUND
Path                 : ../models/anomaly/isolation_forest.joblib
Some model files are missing. Check the paths above.


In [27]:
import os

print("Searching for Logistic Regression model files...\n")

for root, dirs, files in os.walk("../"):
    for file in files:
        if "logistic" in file.lower() and file.endswith(".joblib"):
            print(os.path.join(root, file))

Searching for Logistic Regression model files...

../models\logistic_regression.joblib


In [28]:
import joblib
import os

logistic_path = "../models/logistic_regression.joblib"

logistic_model = joblib.load(logistic_path)

print("Logistic Regression model loaded successfully!")
print("Path:", logistic_path)
print("Model exists:", os.path.exists(logistic_path))
print("Model type:", type(logistic_model))

Logistic Regression model loaded successfully!
Path: ../models/logistic_regression.joblib
Model exists: True
Model type: <class 'sklearn.linear_model._logistic.LogisticRegression'>


In [29]:
import os

model_paths = {
    "Logistic Regression": "../models/logistic_regression.joblib",
    "Random Forest": "../models/supervised/random_forest.joblib",
    "XGBoost": "../models/supervised/xgboost.joblib",
    "LightGBM": "../models/supervised/lightgbm.joblib",
    "Isolation Forest": "../models/anomaly/isolation_forest.joblib"
}

print("FINAL MODEL FILE VERIFICATION")
print("=" * 60)

all_models_exist = True

for model_name, path in model_paths.items():
    exists = os.path.exists(path)

    print(f"{model_name:20} : {'FOUND' if exists else 'MISSING'}")
    print(f"Path                 : {path}")

    if not exists:
        all_models_exist = False

print("=" * 60)

if all_models_exist:
    print("ALL 5 MODELS VERIFIED SUCCESSFULLY!")
    print("PHASE 5 MODEL TRAINING COMPLETED!")
else:
    print("Some model files are missing.")

FINAL MODEL FILE VERIFICATION
Logistic Regression  : FOUND
Path                 : ../models/logistic_regression.joblib
Random Forest        : FOUND
Path                 : ../models/supervised/random_forest.joblib
XGBoost              : FOUND
Path                 : ../models/supervised/xgboost.joblib
LightGBM             : FOUND
Path                 : ../models/supervised/lightgbm.joblib
Isolation Forest     : FOUND
Path                 : ../models/anomaly/isolation_forest.joblib
ALL 5 MODELS VERIFIED SUCCESSFULLY!
PHASE 5 MODEL TRAINING COMPLETED!
